# MPR-Agent on `Qwen3-4B` — Kaggle, local weights

The same four-node agent pipeline as `mpr-agent.ipynb` (`DeepSeek-V4-Flash`),
`mpr-agent-gemma-4-31b-it.ipynb`, `mpr-agent-gpt-oss-120b.ipynb`,
`mpr-agent-gemma3-4b-kaggle.ipynb` and `mpr-agent-qwen3-4b-thinking-kaggle.ipynb`
(its thinking sibling), driving plain `Qwen3-4B` from weights this notebook
**loads onto the GPU itself**. Implementation of Nguyen et al., *"A Graph-Based
Agent Approach to Numerical Reasoning Question Answering"*
([VLSP 2025](https://aclanthology.org/2025.vlsp-1.29/)).

```
q, C ──▶ [1] SubqueryGenerator   G_sq(q, C)            SQ = {sq_1..sq_k}, k∈[3,5]
                    │ fan-out, one independent call per subquery
         [2] SubqueryAnswerer    A_sq(sq_j, C)         V  = {v_1..v_k}
                    │ fan-in
         [3] Planner             P_n-sample(V,C,q,T)   n = 15 candidate plans
                    │
         [4] EquationExtractor   canonicalise → vote → p* → Execute → a*
```

**All the pipeline logic is reused unchanged** — `agentic/agents.py`,
`program.py`, `prompts.py`, `runner.py`, `scoring.py` are cloned from the repo
and imported. This notebook adds one thing: a **transport** that turns "give me
a completion" into a `model.generate()` call on the loaded Qwen3.

## The one thing that is different from the Thinking sibling

Plain `Qwen3-4B` — unlike `Qwen3-4B-Thinking-2507` — never opens `<think>`.
Its chat template goes straight from the prompt to the answer, so every token
budget stays at the paper's own value (no `THINK_HEADROOM`), and there is no
`</think>` span to slice off the decoded output. Everything else in this
notebook — the adaptive chunked n-sampling, the OOM handling, the traceback
clearing, the cross-session resume — is carried over unchanged from the
Thinking sibling, which was written against the same 16 GB T4 for the same
reasons (see section 4 below).

## Kaggle setup

* **Accelerator**: GPU (T4 or P100). Only `cuda:0` is used — a second T4 sits idle.
* **Internet**: must be **ON** (Notebook options) for pip, `git clone`, and the
  Hugging Face download.
* No `HF_TOKEN` needed: `unsloth/Qwen3-4B` is an ungated mirror of `Qwen/Qwen3-4B`,
  and is the exact repo id `qwen3-4b-stf-w-reasoning-trace.ipynb` in this repo
  already loads for SFT on Kaggle.
* **Add Input**: attach your ViNumQA dataset. `/kaggle/input` is read-only, so
  the repo is cloned to `/kaggle/working` instead — see section 2.


## Read this before starting a session: the cost, and the missing baseline

**Cost.** With `use_decomposition=True` and `n_samples=15`, one sample costs
`1 + k + 15` sequential generations (k ≈ 3–5 subqueries) on a single GPU, with
no concurrency available. Unlike the Thinking sibling, there is no prior
same-model measurement in this repo to project from (that notebook's own
~24 s/generation figure is `qwen3-4b-thinking`'s, on a prompt that spends most
of its budget inside `<think>` — it does not transfer here, and guessing would
be worse than leaving it blank). Section 6's smoke run measures your own
s/sample and projects the full 497-sample wall-clock before you commit a
session to it; section 6b is the table to edit if that projection is too long.

A Kaggle session is 12 h. Plan on **not** fitting the full run in one — section
8 resumes across sessions and per-sample checkpointing has always been on.

**Baseline.** This repo has `vsf-vinumqa-0-shot-qwen3-4b.ipynb` and
`vsf-vinumqa-1-shot-qwen3-4b.ipynb`, but — same situation as every local model
in this family — no `*_summary.json` is committed for either in
`0-shot/outputs/` / `1-shot/outputs/`. Section 9's comparison cell will show
`NaN` for both ICL rows unless you run one of those notebooks yourself and save
its summary in the same shape the other models use.

What *is* on record for this model in this repo is a different regime
entirely: the SFT-with-reasoning-trace adapter's own row in the root README.
**That is a fine-tuned checkpoint, not this run** — MPR-Agent trains nothing
and belongs beside the in-context rows above, not the SFT ones.


## Why this notebook defines its own backend instead of using `agentic.backends.LocalBackend`

Unlike the Gemma3 case, `LocalBackend` **would** load this checkpoint
correctly: `Qwen/Qwen3-4B`'s `config.json` names `Qwen3ForCausalLM`, exactly
what `AutoModelForCausalLM` maps, and `MODEL_REGISTRY["Qwen3-4B"]` already
points at it. The reason for a dedicated backend here is not a loading bug —
it is the same VRAM/throughput ceiling every local notebook in this family
runs into on a T4:

```python
# agentic/backends.py, LocalBackend
if n > 1:
    gen_kwargs["num_return_sequences"] = n   # one call, n=15 sequences at once
```

One `generate()` with `num_return_sequences=15` over the planner's ~4k-token
prompt is a certain OOM on a 14.56 GiB card (measured on the Thinking sibling,
same order of magnitude here since it is the same architecture and a similar
prompt length). `LocalBackend`'s own adaptive retry (added after that was
found) halves the batch on OOM — correct, but halving takes 3 straight to 1
and never tries 2, and its `except` clause does not clear the OOM's own
traceback before reclaiming VRAM, so a failed call can leave less free memory
behind than a clean one would. Rather than patch shared package code that four
other notebooks depend on, this notebook reuses the chunked, traceback-clearing
backend already proven out on the Gemma3 and Thinking siblings, retargeted at
this checkpoint.

**What it does reuse from the package**, rather than re-implement:
`build_messages("qwen", ...)` for the message shape — already covered by
`tests/test_backends.py`. There is no `strip_think` here: plain Qwen3 never
opens `<think>`, so the decoded continuation *is* the answer.


## 1. Install

Versions pinned to the ones this repo's other `Qwen3-4B` Unsloth notebooks use
(`qwen3-4b-stf-w-reasoning-trace.ipynb`) — unsloth breaks easily against a
mismatched `transformers`. **Restart the session after this cell** if
`transformers` was already imported in this kernel.


In [ ]:
!pip install unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

## 2. Code and data come from two different places

They have to, because **`/kaggle/input` is mounted read-only**. Anything that
needs writing — the clone, the checkpoints, the results — goes to
`/kaggle/working`.

| | where | why |
|---|---|---|
| **repo** (`agentic/`, `scorer.py`, `.git`) | cloned to `/kaggle/working/...` | needs to be written; `.git` is what `find_project_root()` walks up to |
| **dataset** (`test.json`) | your Kaggle Dataset under `/kaggle/input/...` | attached read-only, never written |

Keeping them as one variable is the mistake to avoid: pointing `ROOT` at the
dataset breaks `import agentic`, breaks the scorer lookup, and makes
`find_project_root()` raise — all three, silently, at different moments.

**Attach your dataset** via *Add Input* in the Kaggle sidebar. A dataset
published as `ldhhieu18/vlsp2025` normally mounts at `/kaggle/input/vlsp2025/`,
**not** at `/kaggle/input/datasets/ldhhieu18/vlsp2025/` — so the cell below does
not hardcode either: set `DATA_PATH` explicitly if you know it, otherwise leave
it `None` and the cell finds the file and tells you the real path.

The cell also checks the schema. A ViNumQA split that can be **scored** needs
`qa.program` and `qa.exe_ans`; `private_test.json` carries `qa.question` only,
so if that is what you attached, PA/EA cannot be computed at all and you want to
know now, not after a multi-session run.

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

# ============================================================== 1. REPO ====
# Cloned into /kaggle/working because it must be WRITABLE: `Runner` writes
# checkpoints under it, and /kaggle/input is read-only (a `git clone` into
# /kaggle/input fails with a permission error).
REPO_URL = "https://github.com/ntphuc149/NumReasoning4VietnameseFinancialText"
REPO_ROOT = Path("/kaggle/working/NumReasoning4VietnameseFinancialText")

if not (REPO_ROOT / ".git").exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_ROOT)], check=True)
else:
    print(f"already cloned: {REPO_ROOT}")

HERE = REPO_ROOT / "notebooks" / "vinumqa" / "graph-agent"
sys.path.insert(0, str(HERE))          # so `import agentic` finds the package

for path in [REPO_ROOT / ".git",
             HERE / "agentic" / "runner.py",
             REPO_ROOT / "notebooks" / "evaluate" / "scorer.py"]:
    assert path.exists(), f"clone incomplete, missing: {path}"
print("repo OK  :", REPO_ROOT)

# ============================================================== 2. DATA ====
# Leave None to auto-discover, or set the path if you already know it, e.g.
#   DATA_PATH = "/kaggle/input/vlsp2025/test.json"
#   DATA_PATH = "/kaggle/input/datasets/ldhhieu18/vlsp2025/test.json"
# Auto-discovery handles both layouts, so None is the safe default: it searches
# /kaggle/input recursively and prints whichever path it actually found.
DATA_PATH = None
DATA_FILENAME = "test.json"

if DATA_PATH is None:
    search_roots = [Path("/kaggle/input")]
    found = [p for root in search_roots if root.exists()
             for p in sorted(root.rglob(DATA_FILENAME))]
    if not found:
        # Fall back to the copy that came with the clone.
        fallback = REPO_ROOT / "datasets" / "ViNumQA" / "origin" / DATA_FILENAME
        assert fallback.exists(), (
            f"no {DATA_FILENAME} under /kaggle/input and none in the clone. "
            f"Attach your dataset via 'Add Input', or set DATA_PATH by hand."
        )
        found = [fallback]
        print(f"no /kaggle/input copy found -- using the repo's own {DATA_FILENAME}")
    if len(found) > 1:
        print(f"several {DATA_FILENAME} found; using the first:")
        for p in found:
            print("   ", p)
    DATA_PATH = found[0]

DATA_PATH = Path(DATA_PATH)
assert DATA_PATH.exists(), f"dataset not found: {DATA_PATH}"
print("dataset  :", DATA_PATH)

# ============================================================ 3. SCHEMA ====
# `load_dataset` takes an absolute path as-is, so DATA_PATH can live anywhere.
with open(DATA_PATH, encoding="utf-8") as handle:
    _samples = json.load(handle)

assert isinstance(_samples, list) and _samples, f"{DATA_PATH} is not a non-empty JSON list"
_first = _samples[0]
_missing_top = {"id", "pre_text", "table", "post_text", "qa"} - set(_first)
assert not _missing_top, f"{DATA_PATH}: samples missing top-level keys {_missing_top}"

_qa = set(_first.get("qa", {}))
SCORABLE = {"program", "exe_ans"} <= _qa
print(f"samples  : {len(_samples)}   qa keys: {sorted(_qa)}")
if SCORABLE:
    print("scorable : yes (qa.program and qa.exe_ans present)")
else:
    print("scorable : NO -- qa carries only", sorted(_qa))
    print("           This looks like a private/unlabelled split. The pipeline")
    print("           will still generate predictions, but PA/EA cannot be")
    print("           computed: every `.score(...)` cell below will be")
    print("           meaningless. Use datasets/ViNumQA/origin/test.json (497")
    print("           labelled samples, shipped with the clone) to measure.")

del _samples

## 3. Load Qwen3-4B

`FastLanguageModel.from_pretrained` with 4-bit weights — the same call this
repo's `qwen3-4b-stf-w-reasoning-trace.ipynb` already makes for this exact
checkpoint on Kaggle: same `model_name`, same `load_in_4bit=True`.

**The VRAM budget.** A T4 has 14.56 GiB. In 4-bit this model holds ~3.2 GiB,
leaving ~11 GiB for everything generation does. Qwen3-4B is 36 layers × 8 KV
heads × 128 head dims, so KV cache costs `2 (K,V) × 36 × 8 × 128 × 2 bytes =
0.14 MB` per token per sequence. The planner prompt is the peak: ~4.2k tokens
of context plus a plain (non-thinking) answer, so well under 5k tokens total —
**~0.7 GiB per sequence** at that length. Three at a time is ~2.1 GiB, which
fits with room to spare; fifteen at a time is ~10.5 GiB, tight against ~11 GiB
free and the certain-enough OOM the backend below starts smaller than and
steps away from.

**`MAX_SEQ_LENGTH = 8192`**, matching the Gemma3 sibling's measurement for this
same paper-prompt shape (0-shot prompts on this dataset run up to ~3.7-4.7k
tokens; the planner's is longer still once the subquery answers are appended).
No extra headroom is needed here — unlike the Thinking sibling, nothing in
this model's output is spent on a reasoning trace before the answer.

`PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True` is set before the torch
import, not decoratively: OOM failures on this card are progressive, each one
leaving less free VRAM than the last, and fragmentation from repeated
variable-size allocations is half of that story. The other half is fixed in
the backend cell.


In [ ]:
# CUDA allocator: expandable segments cut the fragmentation that turns one
# planner OOM into the next one. Must be set BEFORE torch initialises CUDA --
# i.e. before the unsloth import below, which is what pulls torch in.
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

# unsloth must be imported before transformers so its patches take effect.
from unsloth import FastLanguageModel

import gc
import torch

MODEL_REPO = "unsloth/Qwen3-4B"   # ungated mirror of Qwen/Qwen3-4B
MAX_SEQ_LENGTH = 8192              # see the markdown above -- measured, no thinking headroom needed

# 4-bit is the comparable setting for this model in this repo's Unsloth
# notebooks: qwen3-4b-stf-w-reasoning-trace.ipynb loads it the same way. It
# also buys the headroom a chunked 15-sample planner call needs on a T4.
# Flip to False only on a card with >= 24 GiB, and label the row.
LOAD_IN_4BIT = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_REPO,
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit = LOAD_IN_4BIT,
    load_in_8bit = False,
    full_finetuning = False,
)
FastLanguageModel.for_inference(model)

free_bytes, total_bytes = torch.cuda.mem_get_info()
free_gib, total_gib = free_bytes / 2**30, total_bytes / 2**30

print("device:", next(model.parameters()).device)
print("dtype :", next(model.parameters()).dtype)
print(f"VRAM  : {total_gib - free_gib:.1f} GiB held, {free_gib:.1f} GiB free "
      f"of {total_gib:.1f} GiB")

# The planner is the memory peak: SAMPLE_CHUNK sequences, each holding KV
# cache for a ~4-5k-token sequence. Below ~2 GiB free it cannot run even one
# at a time, and every sample will fall back to the direct prompt.
if free_gib < 2.0:
    print()
    print("!" * 78)
    print(f"Only {free_gib:.1f} GiB free -- the planner will OOM even at "
          f"SAMPLE_CHUNK=1.")
    print("Restart the kernel (something else is holding VRAM), or lower")
    print("MAX_SEQ_LENGTH and the token budgets in section 5.")
    print("!" * 78)


## 4. The transport — the only new code in this notebook

`Runner(run_config, client=...)` takes any object with `complete()`,
`sample_n()` and `.usage`; `MultiModelClient` is just the default. Everything in
`agents.py` already calls `self.client.complete(..., model=...)`, so satisfying
that protocol is all it takes to run the unmodified pipeline on local weights.

Five things this has to get right, all carried over from the Thinking sibling,
which solved them against the same card:

* **One `generate()` at a time.** `agents.py` fans out over subqueries and
  n-samples with a `ThreadPoolExecutor` — correct for HTTP, corrupting for a
  single GPU. `_lock` serialises every forward pass. Note the limit of that
  guarantee: it serialises *compute*, not *allocation*. `max_workers_dataset`
  still opens real threads in `Runner.run`, and each in-flight sample holds its
  own tensors on the same card, so that one is set to 1 in the config cell.
* **Chunked n-sampling, adaptively.** `num_return_sequences=15` over a ~4-5k
  prompt is more KV cache than this card comfortably holds at once. The n
  samples are generated in chunks of `SAMPLE_CHUNK`, and the chunk size *steps
  down by one on OOM and stays down*. One at a time, not by halving: halving
  takes 3 straight to 1 and never tries 2. Sticky matters too: retrying the
  large size on every planner call would buy a doomed prefill 497 times over.
  One prompt's KV cache is still shared within each chunk.
* **Cleanup on the failure path.** `gc.collect()` / `empty_cache()` live in a
  `finally` around the single `generate()` call, not after the sampling loop.
  Cleanup that only runs on success is cleanup that never runs when you need it.
* **The OOM's own traceback is cleared before retrying.** Subtle and
  load-bearing. A live traceback holds one frame per level of `generate()`, and
  each of those frames holds its intermediate tensors, so an `empty_cache()`
  taken while the exception is still bound frees nothing at all. `raise ... from
  None` does *not* fix this — it only hides the context from the printed
  traceback while `__context__` keeps the frames alive. `exc.__traceback__ =
  None` in the handler is what actually releases them, and the reclaim is then
  taken outside the `except` block, where `exc` is unbound.
* **Generated tokens leave the GPU immediately.** `row[prompt_len:]` is a
  *view* — it keeps the whole output tensor alive through decoding. `.tolist()`
  copies to host so the VRAM is actually released, the same thing
  `LocalBackend` does.

Plain Qwen3 never opens `<think>`, so unlike the Thinking sibling there is no
reasoning trace to slice off — the decoded continuation *is* the answer.


In [ ]:
import threading

from agentic.backends import build_messages
from agentic.llm import LLMError, Usage

# `torch.OutOfMemoryError` is the torch 2.x name, `torch.cuda.OutOfMemoryError`
# the older alias. Bind whichever exists, so the except clause below cannot
# itself raise AttributeError on a version bump.
OOM_ERROR = getattr(torch, "OutOfMemoryError", torch.cuda.OutOfMemoryError)


class UnslothQwen3Backend:
    """`Backend` protocol over an unsloth-loaded plain Qwen3-4B, for `Runner(client=)`.

    Mirrors `agentic.backends.LocalBackend` in behaviour -- same locking, same
    usage accounting, same budget guard, and the same pure helper
    (`build_messages`) imported rather than copied -- but loads nothing itself:
    it wraps the model/tokenizer already in memory above.

    The one deliberate difference from `LocalBackend`: n-sampling is chunked
    and the chunk size is adaptive. One `num_return_sequences=15` call is right
    on a 40GB+ card and risky on a T4 when each sequence carries real KV cache
    at a ~4-5k-token prompt.
    """

    SAMPLE_CHUNK = 3      # starting n-samples per generate(); steps down on OOM

    def __init__(self, model, tokenizer, config, max_seq_length=MAX_SEQ_LENGTH):
        self.model = model
        self.tokenizer = tokenizer
        self.config = config
        self.max_seq_length = max_seq_length
        self.usage = Usage()
        self._lock = threading.Lock()
        # Adaptive and STICKY: once a batch size OOMs, that size is never tried
        # again this run. Re-attempting 3 on every planner call would pay for a
        # doomed prefill 497 times over.
        self._chunk = self.SAMPLE_CHUNK
        self.oom_backoffs = 0

    # ---------------------------------------------------------------- memory --
    @staticmethod
    def _free_vram():
        gc.collect()
        torch.cuda.empty_cache()

    def _one_call(self, inputs, prompt_len, gen_kwargs, take):
        """Exactly one generate(). Returns token-id LISTS, already off the GPU.

        The `finally` is the point of this method. When `generate()` raises OOM,
        cleanup placed after the sampling loop never runs, so each failure
        leaves its activations resident and the next sample starts with less
        free VRAM than the last.
        """
        out = None
        try:
            call_kwargs = dict(gen_kwargs)
            if take > 1:
                call_kwargs["num_return_sequences"] = take
            with torch.inference_mode():
                out = self.model.generate(**inputs, **call_kwargs)
            # `.tolist()` copies to host. `row[prompt_len:]` on its own is a
            # view that keeps the whole `out` tensor -- and its VRAM -- alive
            # all the way through decoding, which is the other half of the leak.
            return [row[prompt_len:].tolist() for row in out]
        finally:
            del out
            self._free_vram()

    # ------------------------------------------------------------- generate --
    def _generate(self, system, user, max_tokens, temperature, n):
        """The one place that calls generate(). Caller must hold `_lock`."""
        cfg = self.config
        inputs = self.tokenizer.apply_chat_template(
            build_messages("qwen", system, user),
            add_generation_prompt = True,
            tokenize = True,
            return_tensors = "pt",
            return_dict = True,
        ).to(self.model.device)

        try:
            prompt_len = inputs["input_ids"].shape[-1]
            budget = self.max_seq_length - prompt_len
            if budget <= 0:
                raise LLMError(
                    f"prompt ({prompt_len} tokens) exceeds MAX_SEQ_LENGTH="
                    f"{self.max_seq_length}; nothing left to generate. Reload "
                    f"the model with a larger max_seq_length."
                )
            new_tokens = min(max_tokens, budget)

            temp = cfg.temperature if temperature is None else temperature
            gen_kwargs = {"max_new_tokens": new_tokens}
            if temp is not None and temp > 0:
                gen_kwargs.update(do_sample=True, temperature=temp, top_p=cfg.top_p)
                if cfg.send_top_k:
                    gen_kwargs["top_k"] = cfg.top_k
            else:
                gen_kwargs["do_sample"] = False

            texts = []
            remaining = max(1, n)
            while remaining > 0:
                take = min(remaining, self._chunk)
                while True:
                    backed_off = False
                    try:
                        gen_only = self._one_call(inputs, prompt_len, gen_kwargs, take)
                    except OOM_ERROR as exc:
                        # Drop the traceback before anything else. It holds one
                        # frame per level of generate(), and every one of those
                        # frames holds its intermediate tensors -- so an
                        # empty_cache() taken while it is alive frees nothing.
                        # `raise ... from None` does NOT do this: it only hides
                        # the context from the printed traceback, leaving
                        # __context__ (and these frames) referenced.
                        exc.__traceback__ = None
                        if take == 1:
                            # Nothing left to step down to. LLMError rather than
                            # a bare OutOfMemoryError is deliberate: the node
                            # records it, this sample degrades to the
                            # direct-prompt fallback, and the run continues.
                            raise LLMError(
                                f"OOM at batch=1 (prompt={prompt_len} tok, "
                                f"max_new_tokens={new_tokens}). Lower the token "
                                f"budgets in the config cell, or set "
                                f"use_decomposition=False to shorten the "
                                f"planner prompt."
                            ) from None
                        backed_off = True
                    if not backed_off:
                        break
                    # Step down by one, NOT by halving. Halving takes 3 straight
                    # to 1 and never tries 2. Each step down is paid for exactly
                    # once per run, because `_chunk` is sticky.
                    take = max(1, take - 1)
                    self._chunk = take
                    self.oom_backoffs += 1
                    # Outside the except block, so `exc` is unbound and the OOM
                    # is collectable: this reclaim actually reclaims.
                    self._free_vram()
                    print(f"  [oom] SAMPLE_CHUNK -> {take} "
                          f"(prompt={prompt_len} tok)", flush=True)

                self.usage.add(
                    prompt_len,
                    sum(len(g) for g in gen_only),
                    prompt_len + sum(len(g) for g in gen_only),
                )
                texts += [
                    self.tokenizer.decode(g, skip_special_tokens=True).strip()
                    for g in gen_only
                ]
                remaining -= len(gen_only)
            return texts
        finally:
            del inputs
            self._free_vram()

    # --------------------------------------------------------------- public --
    def complete(self, system, user, model, max_tokens, temperature=None):
        with self._lock:
            outputs = self._generate(system, user, max_tokens, temperature, n=1)
        if not outputs or not outputs[0]:
            raise LLMError("generation returned empty output")
        return outputs[0]

    def sample_n(self, system, user, model, n, max_tokens,
                 temperature=None, max_workers=15):
        if n <= 1:
            return [self.complete(system, user, model, max_tokens, temperature)]
        with self._lock:
            outputs = self._generate(system, user, max_tokens, temperature, n=n)
        usable = [o for o in outputs if o]
        if not usable:
            raise LLMError("n-sampling produced no usable output")
        return usable


print("backend defined:", UnslothQwen3Backend.__name__)
print(f"  SAMPLE_CHUNK starts at {UnslothQwen3Backend.SAMPLE_CHUNK}, "
      f"steps down one at a time on OOM, floor 1")


## 5. Configuration

Paper section 5.1, unchanged: `n = 15`, `temperature = 0.6`, `top_p = 0.95`,
`top_k = 20`, Vietnamese prompts, and the paper's own token budgets --
`max_tokens_planner=768` etc. -- with **no headroom added**, unlike the
Thinking sibling. Plain Qwen3-4B answers directly; there is no `<think>` span
eating into the budget before the plan is written.


In [ ]:
import json
import time

import pandas as pd

from agentic import AgentConfig, RunConfig, Runner
from agentic.runner import candidate_diagnostics, load_dataset, revote, score_frame

pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 80)

# The exact key this repo uses for this model everywhere else, so the section-9
# lookups and any results table you paste it into line up.
MODEL = "Qwen3-4B"   # label only -- the backend below is what runs

agent_config = AgentConfig(
    model_subquery_gen=MODEL,
    model_subquery_ans=MODEL,
    model_planner=MODEL,
    model_fallback=MODEL,
    # --- paper section 5.1, unchanged: no thinking headroom needed here ---
    n_samples=15,
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    prompt_lang="vi",
    max_tokens_subquery_gen=1024,
    max_tokens_subquery_ans=512,
    max_tokens_planner=768,
    max_tokens_fallback=512,
    # --- ours; see README.md ---
    vote_mode="canonical",
    use_prompt_ext=False,
    # False, not paper-faithful True: measured on DeepSeek-V4-Flash over the
    # full 497, decomposition OFF beat decomposition ON on both PA and EA (see
    # the "Ablation & prompt-fidelity results" table in README.md) and is this
    # repo's actual default for every other model in the comparison table.
    # Flip to True only to deliberately reproduce the paper-faithful row, and
    # label the result as such -- it is not comparable to the other rows if
    # you do. Untested on this model either way.
    use_decomposition=False,
    # ONE, not the package default of 4. The backend's `_lock` serialises
    # compute, but it does NOT serialise allocation: `Runner.run` really does
    # open a ThreadPoolExecutor, and 4 in-flight samples each hold their own
    # tokenised inputs and their own generate() leftovers on the same card.
    # A thread cannot empty_cache() what another thread is still holding, so
    # 4 workers multiply peak VRAM and fragmentation for no throughput at all.
    max_workers_dataset=1,
    # Irrelevant locally: there is no endpoint to throttle against.
    rpm_limit=None,
    tpm_limit=None,
)

backend = UnslothQwen3Backend(model, tokenizer, agent_config)

run_config = RunConfig(
    dataset_path=str(DATA_PATH),        # absolute -- resolved as-is by load_dataset
    output_dir="notebooks/vinumqa/graph-agent/outputs",
    run_name=f"mpr-agent-{MODEL}-kaggle",
    agent=agent_config,
)

runner = Runner(run_config, client=backend)
# k (subqueries per sample) only exists when use_decomposition=True -- with it
# False (this notebook's default), nodes [1]+[2] never run, so a sample costs
# exactly n_samples generations, not 1 + k + n_samples.
_k = 4 if agent_config.use_decomposition else 0
_gens_per_sample = (1 if agent_config.use_decomposition else 0) + _k + agent_config.n_samples

print(f"MODEL={MODEL!r} -> local weights ({MODEL_REPO}, "
      f"{'4-bit' if LOAD_IN_4BIT else '16-bit'})")
print(f"use_decomposition={agent_config.use_decomposition}  "
      f"n_samples={agent_config.n_samples}")
print(f"generations per sample: {_gens_per_sample}"
      + ("  (1 + k + n_samples, k = subqueries, 3-5)" if agent_config.use_decomposition
         else "  (n_samples only -- decomposition off, nodes [1]+[2] skipped)"))
print(f"planner generate() calls per sample: "
      f"{-(-agent_config.n_samples // backend.SAMPLE_CHUNK)} "
      f"at SAMPLE_CHUNK={backend.SAMPLE_CHUNK}")
print(f"token budgets (paper values, no headroom): "
      f"gen={agent_config.max_tokens_subquery_gen} "
      f"ans={agent_config.max_tokens_subquery_ans} "
      f"plan={agent_config.max_tokens_planner} "
      f"fallback={agent_config.max_tokens_fallback}")
print(runner.graph.describe())


## 6. Smoke run — 10 samples

Deliberately 10, not the 30 the API notebooks use: on local weights this is the
cell that tells you whether the full run is feasible **in this session at all**,
and you want that answer in minutes.

Read these in order, and read the first three *before* looking at PA/EA — a
broken pipeline still prints a complete, plausible-looking summary, because
`equation_extractor` falls back to a plain 0-shot prompt whenever the planner
gives it nothing:

1. `fallback_rate` — the pipeline health signal. `1.0` means **no sample went
   through MPR-Agent at all**; the PA/EA underneath it are then measuring a
   0-shot prompt, not this method. The cell asserts on this.
2. `mean_usable_candidates` — `0.0` means the planner returned nothing to vote
   over. Same story as above, seen from the other side.
3. `sequences/sample` — the raw count of generations, which should be
   `n_samples` = 15 (decomposition is off by default in the config cell above).
   Read this rather than `generate() calls/sample`: because the planner's n
   samples are chunked, one `generate()` call produces `SAMPLE_CHUNK`
   sequences, so the call count in `usage` is only `ceil(n / SAMPLE_CHUNK)` ≈
   5 even in a perfectly healthy run. The cell prints both, with the expected
   value beside each.
4. `oom backoffs` — 0 is ideal. A few is fine, that is the backend finding its
   batch size.
5. `s/sample` — multiply by 497. Compare against section 1's cost note and
   decide which configuration you are actually running before section 8.
6. `empty_rate` — once the four above are healthy, this one is about the model:
   it means the model emitted something unparseable, which no runtime fix
   touches.
7. `mean_consensus` — if it is 1.0, the 15 samples collapsed to one program and
   `n_samples` is pure cost, exactly as measured on `gemma-4-31B-it`. Worth
   checking before paying for n=15 over 497 samples.


In [ ]:
smoke_runner = Runner(
    RunConfig(**{**run_config.__dict__, "run_name": "kaggle-smoke10", "limit": 10}),
    client=backend,
)

started = time.time()
smoke_df = smoke_runner.run(show_progress=True)
elapsed = time.time() - started

smoke_scored, smoke_summary = smoke_runner.score(smoke_df)
usage = smoke_summary["usage"]
n = len(smoke_df)

for key, value in smoke_summary.items():
    print(f"{key:>24}: {value}")
print()
print(f"{'seconds/sample':>24}: {elapsed / n:.1f}")
print(f"{'tokens/sample':>24}: {usage['total_tokens'] / n:.0f}")
# Two different numbers, and confusing them is how a healthy run gets
# diagnosed as broken. `usage.requests` counts generate() CALLS, and the
# planner's 15 samples arrive `SAMPLE_CHUNK` at a time -- so a healthy run
# shows ~5 calls, not ~15. `backend` here does not track sequence counts
# itself (no thinking-budget health signal needed), so both expectations are
# computed from `agent_config` alone.
# k (subqueries) only exists when use_decomposition=True -- see the config
# cell. With it False (this notebook's default), a sample is n_samples
# generate() calls/sequences only, not 1 + k + n_samples.
_k = 4 if agent_config.use_decomposition else 0
_base = 1 if agent_config.use_decomposition else 0
expected_calls = _base + _k + -(-agent_config.n_samples // backend._chunk)
expected_seqs = _base + _k + agent_config.n_samples
print(f"{'generate() calls/sample':>24}: {usage['requests'] / n:.1f} "
      f"(expect ~{expected_calls} at SAMPLE_CHUNK={backend._chunk})")
print(f"{'oom backoffs':>24}: {backend.oom_backoffs} "
      f"(SAMPLE_CHUNK now {backend._chunk})")

# ---------------------------------------------------- pipeline health gate --
# `fallback_rate` is the honest signal: 1.0 means every sample skipped the
# entire MPR-Agent pipeline and answered from a plain 0-shot prompt, so PA/EA
# above are measuring nothing at all. That is what a planner that always raises
# looks like from out here, and it is indistinguishable from "the model is bad"
# unless you check it.
healthy = (
    smoke_summary["fallback_rate"] <= 0.5
    and smoke_summary["mean_usable_candidates"] > 0
)
if not healthy:
    print("\n" + "!" * 78)
    print("PIPELINE UNHEALTHY -- do NOT start the 497-sample run.")
    print(f"  fallback_rate          = {smoke_summary['fallback_rate']} "
          f"(want <= 0.5)")
    print(f"  mean_usable_candidates = {smoke_summary['mean_usable_candidates']} "
          f"(want > 0)")
    print(f"  generate() calls/sample = {usage['requests'] / n:.1f} "
          f"(want ~{expected_calls})")
    print("Run the diagnostic cell below before changing anything.")
    print("!" * 78)
else:
    print("pipeline healthy: the planner is producing candidates.")

full_n = len(load_dataset(run_config.dataset_path))
projected_h = elapsed / n * full_n / 3600
print(f"\nprojected for {full_n} samples: {projected_h:.1f} h")
if projected_h > 9:
    print(f"  -> will NOT fit one 12h Kaggle session "
          f"(~{projected_h / 8:.1f} sessions at SESSION_BUDGET_H=8).")
    print("     Either pick a cheaper configuration below, or plan on resuming")
    print("     across sessions: checkpointing is per sample, so re-running the")
    print("     full-run cell in a new session skips everything already done.")


### 6b. If the projection is too long — cheaper configurations

Ordered by how much they cost you scientifically, cheapest concession first.
Each is one line; re-run the config cell after editing.

| change | generations/sample | what you lose |
|---|---:|---|
| `n_samples=5` | 5 | vote resolution. Harmless if `mean_consensus` came back 1.0 above; real loss if it did not. |
| `limit=150` in `RunConfig` | unchanged | comparability — a 150-sample number is **not** the same metric as this repo's other 497-sample rows, and must be labelled as such wherever you report it. |

Do **not** reach for `limit` first. A partial run on the full method is a weaker
result than a complete run on a cheaper method, because the latter is still a
clean row in the ablation table the paper's Table 4 defines.


## 7. Read one trace end to end

The point of an agent pipeline over a single prompt is that every intermediate
step is inspectable. Read a few by hand before trusting any aggregate.


In [ ]:
with open(smoke_runner.checkpoint_path, encoding="utf-8") as handle:
    traces = json.load(handle)

gold_lookup = {str(s["id"]): s["qa"] for s in load_dataset(run_config.dataset_path)}


def show(record):
    gold = gold_lookup.get(record["id"], {})
    print("=" * 100)
    print(f"id       : {record['id']}")
    print(f"question : {record['question']}")
    print(f"\n[1] subqueries ({len(record['subqueries'])}):")
    for item in record["subqueries"]:
        print(f"      - {item}")
    print("\n[2] answers:")
    for item in record["subquery_answers"]:
        print(f"      * {item['answer'][:150]}")
    candidates = record.get("candidates", [])
    distinct = {c["program"] for c in candidates if c["program"]}
    print(f"\n[3] {len(candidates)} plans sampled -> {len(distinct)} distinct program(s)")
    for program in list(distinct)[:5]:
        print(f"      {program}")
    vote_info = record.get("vote") or {}
    print(
        f"\n[4] clusters={vote_info.get('n_clusters')} "
        f"consensus={vote_info.get('consensus')} fallback={record.get('fallback')}"
    )
    print(f"      p*   : {record['program']}")
    print(f"      gold : {gold.get('program')}")
    print(f"      a*   : {record['answer']}   gold: {gold.get('exe_ans')}")


for record in traces[:3]:
    show(record)

### Where candidates die

Each generated plan either becomes a program or fails at a named stage: `parse`
(not a plan), `transpile` (a plan with no ViNumQA equivalent), `row_lookup`
(`table_*` naming a row the table does not have), or `execute`.

Compare against the hosted models, where `ok` dominates and the interesting
failures are `transpile` and `row_lookup`.


In [ ]:
from collections import Counter

diagnostics = candidate_diagnostics(smoke_runner.checkpoint_path)

if not diagnostics.empty:
    print(diagnostics["stage"].value_counts(normalize=True).round(4).to_string())
    print("\nmost common failures:")
    print(diagnostics[diagnostics["stage"] != "ok"]["error"]
          .value_counts().head(10).to_string())
else:
    # `candidate_diagnostics` builds one row per generated candidate. An empty
    # frame means NO sample produced a single candidate, i.e. the planner never
    # returned a usable plan -- so there is nothing to break down by stage.
    # Indexing the empty frame for "stage" is the `KeyError: 'stage'` you would
    # otherwise get here; the real diagnosis is in the trace file already.
    raw = json.loads(Path(smoke_runner.checkpoint_path).read_text(encoding="utf-8"))
    print(f"NO CANDIDATES across {len(raw)} sample(s).")
    print("The planner returned no usable plan, so there is no stage breakdown.")
    print("This is a pipeline failure, not a model-quality result -- diagnose it")
    print("before reading anything else in this notebook.\n")

    failed = Counter()
    ok_nodes = Counter()
    for record in raw:
        for trace in record.get("traces", []):
            (failed if not trace.get("ok", True) else ok_nodes)[trace["node"]] += 1

    print(f"{'node':<24}{'ok':>6}{'raised':>8}")
    for node in ("subquery_generator", "subquery_answerer", "planner",
                 "equation_extractor"):
        if ok_nodes[node] or failed[node]:
            print(f"{node:<24}{ok_nodes[node]:>6}{failed[node]:>8}")

    messages = Counter(e for record in raw for e in record.get("errors", []))
    if messages:
        print("\nrecorded errors (most common first):")
        for message, count in messages.most_common(8):
            print(f"  [{count}x] {message[:200]}")
    else:
        print("\nNo node raised: the planner returned text, but every plan")
        print("failed to parse into a program. Inspect the raw output directly:")
        print("  raw[0]['traces']  ->  per-node detail")

    blob = " ".join(messages)
    print("\nWhat to check, in order:")

    if "OutOfMemory" in blob or "OOM" in blob:
        print("  >> CUDA OUT OF MEMORY. This is a VRAM problem, not a prompt or")
        print("     a model problem. The planner is the peak: SAMPLE_CHUNK")
        print("     sequences, each carrying real KV cache for the longest")
        print("     prompt in the pipeline.")
        print(f"     Backend state: SAMPLE_CHUNK={backend._chunk}, "
              f"{backend.oom_backoffs} backoff(s) so far.")
        if backend._chunk > 1:
            print("     -> It has not bottomed out yet. Re-run the smoke cell;")
            print("        the shrunk chunk size is sticky and may already be")
            print("        enough.")
        else:
            print("     -> Already at batch=1 and still OOM. In order:")
            print("        1. use_decomposition=False in the config cell, which")
            print("           drops the subquery answers from the planner")
            print("           prompt and shortens the peak prompt outright.")
            print("        2. MAX_SEQ_LENGTH = 8192 (or lower) and restart the")
            print("           kernel -- this also caps how much KV cache a")
            print("           single generate() call can grow to.")
        print("     Also confirm max_workers_dataset=1: extra Runner threads")
        print("     multiply peak VRAM without buying any throughput here.")
    else:
        print("  1. Prompt length. This backend raises LLMError when the prompt")
        print("     alone exceeds MAX_SEQ_LENGTH. The planner prompt is the")
        print("     longest in the pipeline (context + every subquery answer).")
        print("     If the errors above say 'exceeds MAX_SEQ_LENGTH', reload")
        print("     with a larger max_seq_length, or set")
        print("     use_decomposition=False to drop the subquery answers.")
        print("  2. The system turn. The planner and the fallback are the only")
        print("     nodes that send a system message; the two subquery nodes")
        print("     send system=None. If those two succeeded and only the")
        print("     planner raised, suspect the chat template's system-role")
        print("     handling.")
        print("  3. Empty generations. 'no usable output' means every sample")
        print("     decoded to an empty string. Raise max_tokens_planner.")

    print("\nOne planner prompt, end to end, reproduces it in isolation:")
    print("  from agentic.agents import build_state")
    print("  s = build_state(load_dataset(run_config.dataset_path, limit=1)[0])")
    print("  p = runner.graph.nodes['planner']")
    print("  print(backend.complete(p.prompts.planner_system,")
    print("                         p.build_user_prompt(s), MODEL,")
    print("                         agent_config.max_tokens_planner))")


## 8. Full run — `test.json`, 497 samples, across as many sessions as it takes

**Per-sample checkpointing is already on and always was.**
`RunConfig.checkpoint_every` defaults to `1`, so `Runner` rewrites its traces
file after *every* completed sample, atomically (`.tmp` then `replace`). A CUDA
OOM, a dead kernel, a `KeyboardInterrupt` — none of them can cost you more than
the one sample that was in flight. That part needs no fixing.

What the cell below adds is the thing per-sample checkpointing does *not*
solve — and on this model it is not optional. At the projection section 6 gave
you, the paper-faithful configuration needs **six or more Kaggle sessions**, so
"resume" has to mean *across* sessions:

* **Surviving the end of a session.** The checkpoint lives inside the clone at
  `/kaggle/working/NumReasoning4VietnameseFinancialText/notebooks/vinumqa/graph-agent/outputs/`,
  and the clone is re-created from scratch every session. The cell searches
  `/kaggle/input` for a traces file from a previous run and seeds the checkpoint
  from it before starting.
* **Stopping before the wall, not at it.** Kaggle terminates at 12 h with no
  warning and no opportunity to save. `SESSION_BUDGET_H` stops the run
  deliberately, with time left to save the output. It also refuses to *start* a
  chunk it does not expect to finish inside the budget, using this session's own
  measured s/sample — so it stops early rather than being killed mid-sample.
  `CHUNK_SIZE = 5` bounds the overshoot at about `5 × s/sample`, which on this
  model is ~40 minutes; drop it to 2 if your budget is tight.
* **A stable place to find the file.** Everything is mirrored to
  `/kaggle/working/checkpoints/` after every chunk. Top level, obvious in the
  Output tab, no digging through the clone.
* **Not lying about a partial run.** If it stops early it scores only the
  samples that actually ran — feeding the scorer all 497 with 40 done would mark
  457 empty predictions wrong and report a PA that is about nothing — and it
  writes to a `-partial-NNofMM` filename rather than the canonical
  `{run_name}_summary.json` that the comparison cell further down reads.

### The loop, concretely

1. Run every cell top to bottom. This one stops itself after `SESSION_BUDGET_H`.
2. **Save Version** (or download `/kaggle/working/checkpoints/`) before the
   session ends. Nothing in `/kaggle/working` survives otherwise.
3. New session: **Add Input →** this notebook's saved output, then run every
   cell down to here again. It finds the traces file, prints `restored
   checkpoint from ...`, and picks up at the next unfinished sample.
4. Repeat until it prints `RUN COMPLETE`, at which point it writes the real
   `_results.csv` and `_summary.json`.

`SESSION_BUDGET_H = 8.0` is a starting point, not a measurement — set it to
however long you actually intend to babysit the session, leaving margin to save.

In [ ]:
import io
import re
import shutil

# ============================================================== knobs =======
# Stop cleanly after this long. Kaggle kills the session at 12h with no warning
# and no chance to save, so leave real margin: the cell below also refuses to
# start a chunk it does not expect to finish in time.
SESSION_BUDGET_H = 8.0
# Stop-check granularity. Checkpointing is per-sample regardless
# (`RunConfig.checkpoint_every == 1`); this only bounds how far past the budget
# the run can overshoot -- about `CHUNK_SIZE x s/sample`, which is ~40 min here.
CHUNK_SIZE = 5
# Mirrored here after every chunk. The canonical checkpoint lives inside the
# clone, which is easy to lose track of in the Output tab; this is a stable
# top-level path to download from.
MIRROR_DIR = Path("/kaggle/working/checkpoints")

# =========================================== restore a previous session =====
# If a previous session's traces file was attached as a Kaggle input (Add Input
# -> your earlier notebook version's output), seed the checkpoint from it. The
# clone is fresh every session, so without this the run restarts from zero.
MIRROR_DIR.mkdir(parents=True, exist_ok=True)
if not runner.checkpoint_path.exists():
    kaggle_input = Path("/kaggle/input")
    found = (sorted(kaggle_input.rglob(runner.checkpoint_path.name))
             if kaggle_input.exists() else [])
    if found:
        shutil.copy2(found[0], runner.checkpoint_path)
        print(f"restored checkpoint from {found[0]}")
    else:
        print("no previous checkpoint found -- starting fresh")

# ================================================================ state =====
all_samples = load_dataset(run_config.dataset_path)
runner._load_checkpoint()
done_ids = set(runner._results)
todo = [s for s in all_samples if str(s.get("id", "")) not in done_ids]

print(f"checkpoint : {runner.checkpoint_path}")
print(f"mirror     : {MIRROR_DIR}")
print(f"done       : {len(done_ids)}/{len(all_samples)}")
print(f"remaining  : {len(todo)}")
print(f"budget     : {SESSION_BUDGET_H:.1f} h\n")


# `Runner.run` prints a banner on every call; at one call per chunk that is 100
# lines of noise. Drop those two lines only -- the backend's [oom] and [clamp]
# warnings go to the same stream and must still get through.
_NOISE = re.compile(r"^(?:\d+ sample\(s\) total|Resumed \d+ sample\(s\))")


class _FilteredStdout(io.TextIOBase):
    def __init__(self, target):
        self.target, self._buf = target, ""

    def write(self, text):
        self._buf += text
        while "\n" in self._buf:
            line, self._buf = self._buf.split("\n", 1)
            if not _NOISE.match(line):
                self.target.write(line + "\n")
        return len(text)

    def flush(self):
        self.target.flush()


# ================================================================== run =====
deadline = time.time() + SESSION_BUDGET_H * 3600
session_started = time.time()
done_this_session = 0
stopped_early = False
stop_reason = ""

try:
    from tqdm.auto import tqdm
    bar = tqdm(total=len(todo), desc=run_config.run_name, unit="sample")
except ImportError:
    bar = None

real_stdout = sys.stdout
try:
    for start in range(0, len(todo), CHUNK_SIZE):
        chunk = todo[start:start + CHUNK_SIZE]

        # Do not start a chunk that will not finish in time -- overshooting the
        # budget is how you get killed mid-sample with the session unsaved.
        if done_this_session:
            per_sample = (time.time() - session_started) / done_this_session
            if time.time() + per_sample * len(chunk) > deadline:
                stopped_early = True
                stop_reason = f"budget of {SESSION_BUDGET_H:.1f} h reached"
                break
        elif time.time() > deadline:
            stopped_early = True
            stop_reason = "budget already spent"
            break

        sys.stdout = _FilteredStdout(real_stdout)
        try:
            runner.run(
                samples=chunk,
                show_progress=False,
                on_result=(lambda record: bar.update(1)) if bar else None,
            )
        finally:
            sys.stdout = real_stdout

        done_this_session += len(chunk)
        shutil.copy2(runner.checkpoint_path, MIRROR_DIR / runner.checkpoint_path.name)

        if bar is not None:
            per_sample = (time.time() - session_started) / done_this_session
            left = len(todo) - done_this_session
            bar.set_postfix(
                s_per_sample=f"{per_sample:.0f}",
                eta_h=f"{per_sample * left / 3600:.1f}",
                budget_h_left=f"{max(0.0, deadline - time.time()) / 3600:.1f}",
                oom_backoffs=backend.oom_backoffs,
            )
except KeyboardInterrupt:
    stopped_early = True
    stop_reason = "interrupted by hand"
finally:
    sys.stdout = real_stdout
    if bar is not None:
        bar.close()
    if runner.checkpoint_path.exists():
        shutil.copy2(runner.checkpoint_path, MIRROR_DIR / runner.checkpoint_path.name)

# ================================================================ score =====
runner._load_checkpoint()
done_samples = [s for s in all_samples if str(s.get("id", "")) in runner._results]
complete = len(done_samples) == len(all_samples)

# Score only what actually ran. Feeding `to_dataframe` the full 497 while 40 are
# done would score 457 empty predictions as wrong and report a PA that is not
# about this model at all.
df = runner.to_dataframe(done_samples)
scored, summary = runner.score(df)

print(f"\nsession: {(time.time() - session_started) / 3600:.2f} h, "
      f"{done_this_session} sample(s) done here")
print(f"total  : {len(done_samples)}/{len(all_samples)}")
print(f"oom backoffs this session: {backend.oom_backoffs} "
      f"(SAMPLE_CHUNK now {backend._chunk})\n")

for key, value in summary.items():
    print(f"{key:>24}: {value}")

if complete:
    results_path, summary_path = runner.save(scored, summary)
    for path in (results_path, summary_path):
        shutil.copy2(path, MIRROR_DIR / path.name)
    print(f"\nRUN COMPLETE -- all {len(all_samples)} samples.")
    print(f"saved: {results_path}\n       {summary_path}")
    print(f"mirrored to {MIRROR_DIR}")
else:
    # Deliberately NOT runner.save(): that writes the canonical
    # `{run_name}_summary.json` which the comparison cell below reads, and a
    # partial number sitting at that path would be indistinguishable from the
    # 497-sample row it is not.
    stem = f"{run_config.run_name}-partial-{len(done_samples)}of{len(all_samples)}"
    partial_csv = runner.output_dir / f"{stem}_results.csv"
    scored.drop(columns=["table_raw"]).to_csv(partial_csv, index=False)
    shutil.copy2(partial_csv, MIRROR_DIR / partial_csv.name)

    print(f"\n{'=' * 78}")
    print(f"STOPPED EARLY: {stop_reason}")
    print(f"The numbers above cover {len(done_samples)} of {len(all_samples)} "
          f"samples. They are NOT the 497-sample row -- do not report them as one.")
    print(f"\nTo continue in a new session:")
    print(f"  1. Save Version (or download) so this session's output is kept.")
    print(f"     The file that matters is:")
    print(f"       {MIRROR_DIR / runner.checkpoint_path.name}")
    print(f"  2. In the new session: Add Input -> this notebook's output.")
    print(f"  3. Run every cell down to and including this one. It finds that")
    print(f"     traces file under /kaggle/input by itself and resumes at "
          f"sample {len(done_samples) + 1}.")
    remaining = len(all_samples) - len(done_samples)
    if done_this_session:
        per_sample = (time.time() - session_started) / done_this_session
        print(f"\n{remaining} left, ~{per_sample:.0f} s/sample "
              f"-> ~{per_sample * remaining / 3600:.1f} h "
              f"(~{per_sample * remaining / 3600 / SESSION_BUDGET_H:.1f} more "
              f"sessions at this budget)")
    print("=" * 78)

### Reading the results frame — which column is the prediction

`scored` and the saved `*_results.csv` follow the column convention
`notebooks/evaluate/scorer.py` expects, and it is **not** the intuitive one:

| column | what it holds |
|---|---|
| `program`, `answer` | **gold**, straight from the dataset |
| `generated_program`, `generated_answer` | **the model's prediction** |
| `pa_score`, `ea_score` | per-row grades |

So `df[["id", "program", "answer"]]` shows you the *dataset*, not your run — it
looks like a plausible list of predictions and will happily convince you the
model nailed every sample. Use `generated_program` / `generated_answer` for
anything you intend to read, quote, or eyeball:

```python
scored[["id", "program", "generated_program", "pa_score", "ea_score"]].head(20)
```

The `show()` trace helper above already reads the checkpoint, where
`record["program"]` **is** the prediction and gold is looked up separately — so
that cell is safe to read as-is. This warning is about the DataFrame and CSV
only.

## 9. Against this repo's existing `Qwen3-4B` rows

Same model, same test set, same scorer — the only thing that changes is the
architecture. This mirrors the paper's Table 2. Rows are read from whatever this
repo has actually saved for this model, so a missing baseline shows as `NaN`
rather than another model's number standing in for it.

**Both ICL rows will likely be `NaN`.** `vsf-vinumqa-0-shot-qwen3-4b.ipynb` and
`vsf-vinumqa-1-shot-qwen3-4b.ipynb` exist, but no `*_summary.json` is committed
for this model in `0-shot/outputs/` or `1-shot/outputs/` as of this writing.
**Do not fill these rows from memory or from the SFT table**; re-run one of
those two notebooks and save its summary to
`0-shot/outputs/0shot_Qwen3-4B_summary.json` (same shape the other models use),
then re-run this cell.

Note that `Qwen3-4B` is one of the three models this repo *fine-tunes*, so its
SFT and STaNR rows in the root README are a different comparison entirely —
MPR-Agent trains nothing, and belongs beside the in-context rows above.


In [ ]:
ICL_SETTINGS = {
    "0-shot": REPO_ROOT / f"notebooks/vinumqa/0-shot/outputs/0shot_{MODEL}_summary.json",
    "1-shot": REPO_ROOT / f"notebooks/vinumqa/1-shot/outputs/1shot_{MODEL}_summary.json",
}

rows = []
for label, path in ICL_SETTINGS.items():
    entry = {"method": f"{MODEL}, {label}", "PA": None, "EA": None}
    if path.exists():
        with open(path, encoding="utf-8") as handle:
            blob = json.load(handle)
        entry["PA"] = round(blob["program_accuracy"], 4)
        entry["EA"] = round(blob["execution_accuracy"], 4)
    else:
        print(f"no saved summary for {label}: {path.relative_to(REPO_ROOT)}")
    rows.append(entry)

# A resumed run may still be partway through. The ICL rows above are full
# 497-sample numbers, so putting an unlabelled 40-sample number next to them in
# the same table is the one way this notebook could still mislead you after the
# run cell warned. Tag it in the row name itself, where it travels with the
# number if the table gets copied out.
suffix = "" if complete else f"  [PARTIAL {summary['n']}/{len(all_samples)}]"

rows += [
    {
        "method": f"{MODEL}, MPR-Agent{suffix}",
        "PA": round(summary["program_accuracy"], 4),
        "EA": round(summary["execution_accuracy"], 4),
    },
    {
        "method": f"{MODEL}, MPR-Agent (oracle@{agent_config.n_samples}){suffix}",
        "PA": round(summary["oracle_pa"], 4),
        "EA": round(summary["oracle_ea"], 4),
    },
]

if not complete:
    print(f"WARNING: the two MPR-Agent rows cover {summary['n']} of "
          f"{len(all_samples)} samples and are NOT comparable with the "
          f"0-/1-shot rows above.")

pd.DataFrame(rows)

## 10. Re-vote offline — free, no GPU time

Every candidate is kept on disk with its program and executed value
(`keep_all_candidates`), so changing *how the winner is chosen* costs nothing.
On this run that matters more than anywhere else in the repo: re-running the
pipeline is days of your own GPU, while re-voting is seconds of CPU.

In [ ]:
samples = load_dataset(run_config.dataset_path)
rows = []
for mode in ("canonical", "symbolic"):
    _, mode_summary = score_frame(revote(runner.checkpoint_path, samples, mode=mode))
    rows.append(
        {
            "vote_mode": mode,
            "PA": round(mode_summary["program_accuracy"], 4),
            "EA": round(mode_summary["execution_accuracy"], 4),
        }
    )
pd.DataFrame(rows)

## What was and was not verified before you ran this

Stated plainly so you know where to look first if something breaks.

**Verified without a GPU** (read from this repo, before this notebook was
adapted from its Thinking sibling):

* `unsloth/Qwen3-4B`, `FastLanguageModel.from_pretrained`, `load_in_4bit=True`
  and `transformers==4.56.2` / `trl==0.22.2` are exactly what
  `sft-w-reasoning-trace-distill/qwen3-4b-stf-w-reasoning-trace.ipynb` already
  uses for this checkpoint on Kaggle. Nothing here is a new loading recipe.
* `build_messages` is imported from `agentic/backends.py`, not re-implemented,
  and is already covered by `tests/test_backends.py`.
* `Runner(run_config, client=...)` accepts an arbitrary client and passes it
  straight to `build_default_graph`, and `agents.py` only ever calls
  `client.complete(...)` and `client.sample_n(...)` — the two methods this
  backend implements.
* The chunked-batch OOM handling and traceback-clearing fix are carried over
  verbatim from the Thinking sibling (`mpr-agent-qwen3-4b-thinking-kaggle.ipynb`)
  and the Gemma3 sibling before it, both already exercised against a real T4.

**Not verified** — no GPU was available when this was adapted:

* The actual `from_pretrained` call and anything about VRAM for *this specific*
  checkpoint (as opposed to the Thinking one). `SAMPLE_CHUNK = 3` is carried
  over from the siblings' arithmetic, not re-measured for plain Qwen3-4B's
  (shorter, non-thinking) output length; it steps itself down on OOM regardless.
* Every runtime number. There is no same-model measurement in this repo to
  project from — see section 1. Use your own smoke-run projection, not a
  number quoted from the Thinking or Gemma3 siblings.

**If the load cell fails**, the pinned versions are the first suspect — Kaggle
updates its base image and `transformers==4.56.2` may need moving in step with
whatever unsloth currently requires. Check Unsloth's own current Qwen3 notebook
(<https://unsloth.ai/docs/get-started/unsloth-notebooks>) and match it, then
update the pins here and in the two `qwen3-4b-*` SFT notebooks together.
